
# 🔥 Wildfire Decision Support System – Environmental Intelligence Platform

A machine-learning based platform for wildfire risk assessment, environmental monitoring and automated alert generation across Salta Province.

### 🌲 Key Features

📍 Geospatial risk assessment and mapping

🔥 Active fire detection using NASA FIRMS satellite data

🌦️ Weather forecast integration and environmental indicators

🤖 Automated Telegram-based alert distribution

📊 Interactive visualization and monitoring dashboards

🧠 Predictive wildfire risk modeling with machine learning


### ⚙️ Tech Stack

Python · Pandas · NumPy · XGBoost · Folium · Matplotlib · OpenWeather API · NASA FIRMS · Telegram Bot API


### 🎯 Project Goal

To improve wildfire preparedness and response by transforming environmental and geospatial data into actionable intelligence for decision-makers.

### 🌎 Deployment Context

Salta Province, Argentina

---

**Author:** Martin Bielke

## 🧠 Wildfire Risk Prediction Model

To support early warning capabilities, a supervised machine learning model is trained on historical wildfire, weather and environmental data.

The model learns patterns associated with wildfire occurrence and generates risk estimates for monitored areas. These predictions are later combined with geospatial visualization, environmental indicators and alerting workflows to provide actionable intelligence for decision-makers.

**Techniques used**

- Feature engineering
- Supervised machine learning
- XGBoost classification
- Model validation and performance evaluation
- Probability-based risk assessment

In [ ]:
"""
Entrenamiento del modelo XGBoost
- Top 50 celdas, balance hídrico, FWI, humedad, gradientes de presión, altitud.
- Climatologías calculadas desde train (sin fuga).
"""

import pandas as pd
import numpy as np
import xarray as xr
import requests
import geopandas as gpd
from pathlib import Path
from sklearn.metrics import roc_auc_score, precision_score, recall_score, average_precision_score
from sklearn.neighbors import BallTree
import xgboost as xgb
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

# ========================= CONFIGURACIÓN =========================
ERA5_DIR  = Path("era5_salta")
RAYOS_NC  = Path("lightning/LISOTD_HRAC_V2.3.2015.nc")
EXCEL_POB = Path("data/population/Indicadores de personas. Radios, 2022 - Salta.xlsx")
RANDOM_STATE = 42
TOP_N = 50

ENRIQUECER = True
CONDICIONES_EXTREMAS = {
    't2m_max': 35,
    'dias_sin_lluvia': 10,
    'wind_max': 7,
    'swvl1_aprox': 0.10,
}
FACTOR_DUPLICACION = 3
# =================================================================

# -------------------- Funciones auxiliares --------------------
def cargar_limite_salta():
    limite_path = ERA5_DIR / "salta_provincia.gpkg"
    if not limite_path.exists():
        print("Descargando límite de Salta...")
        import osmnx as ox
        salta = ox.geocode_to_gdf("Provincia de Salta, Argentina")
        salta = salta.to_crs("EPSG:4326")
        salta.to_file(limite_path, driver='GPKG')
    return gpd.read_file(limite_path)

def obtener_departamento(lat, lon, cache={}):
    clave = (round(lat, 2), round(lon, 2))
    if clave in cache:
        return cache[clave]
    url = f"https://apis.datos.gob.ar/georef/api/ubicacion?lat={lat}&lon={lon}"
    try:
        response = requests.get(url, timeout=3)
        data = response.json()
        if 'ubicacion' in data and 'departamento' in data['ubicacion']:
            depto = data['ubicacion']['departamento']['nombre']
            cache[clave] = depto
            return depto
    except:
        pass
    cache[clave] = "Desconocido"
    return "Desconocido"

def obtener_feriados_argentina(anio):
    url = f"https://date.nager.at/api/v3/PublicHolidays/{anio}/AR"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return [h['date'] for h in response.json()]
    except Exception as e:
        print(f"⚠️ Error en feriados {anio}: {e}")
        return []

# -------------------- 1. Cargar datos base y top celdas --------------------
print("Cargando datos base...")
fired = pd.read_parquet(ERA5_DIR / "fired_completo.parquet")
negat = pd.read_parquet(ERA5_DIR / "negativos.parquet")
limite_salta = cargar_limite_salta()

gdf_fired = gpd.GeoDataFrame(
    fired,
    geometry=gpd.points_from_xy(fired['lon_era5'], fired['lat_era5']),
    crs="EPSG:4326"
)
dentro = gpd.sjoin(gdf_fired, limite_salta[['geometry']], how='inner', predicate='within')
fired_salta = fired.loc[dentro.index].copy()
print(f"Incendios totales: {len(fired)} | Dentro de Salta: {len(fired_salta)}")

frecuencia = fired_salta.groupby(['lat_era5', 'lon_era5']).size().reset_index(name='frecuencia')
top_celdas = frecuencia.sort_values('frecuencia', ascending=False).head(TOP_N).copy()
print(f"Top {TOP_N} celdas en Salta:")
print(top_celdas.head(10))

print("Obteniendo departamentos...")
departamentos = []
for _, row in top_celdas.iterrows():
    depto = obtener_departamento(row['lat_era5'], row['lon_era5'])
    departamentos.append(depto)
    time.sleep(0.2)
top_celdas['departamento'] = departamentos
top_celdas.to_parquet(ERA5_DIR / "top_celdas.parquet", index=False)
print("✅ top_celdas.parquet guardado")

# Umbrales logarítmicos
log_min = np.log(top_celdas['frecuencia'].min())
log_max = np.log(top_celdas['frecuencia'].max())
top_celdas['umbral_celda'] = 0.45 - 0.15 * ((np.log(top_celdas['frecuencia']) - log_min) / (log_max - log_min))
top_celdas[['lat_era5', 'lon_era5', 'umbral_celda']].to_parquet(ERA5_DIR / "umbrales_celda.parquet", index=False)
print("✅ umbrales_celda.parquet guardado")

# -------------------- 2. Altitud de cada celda --------------------
print("Obteniendo altitud de cada celda...")
def obtener_altitud(lat, lon):
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()['elevation'][0]

altitudes = []
for _, row in top_celdas.iterrows():
    alt = obtener_altitud(row['lat_era5'], row['lon_era5'])
    altitudes.append(alt)
    print(f"({row['lat_era5']}, {row['lon_era5']}) -> {alt} m")
    time.sleep(0.1)
top_celdas['altitud'] = altitudes
top_celdas[['lat_era5', 'lon_era5', 'altitud']].to_parquet(ERA5_DIR / "altitud_celdas.parquet", index=False)
print("✅ altitud_celdas.parquet guardado")

# -------------------- 3. Construir dataset solo con top celdas --------------------
pos = fired_salta[['lat_era5', 'lon_era5', 't2m_max', 't2m_mean', 't2m_anomalia',
                   'wind_max', 'wind_dir', 'precip', 'dias_sin_lluvia', 'sp_min',
                   'sp_delta', 'swvl1', 'ndvi', 'ndvi_anomalia', 'date']].copy()
pos['fuego'] = 1
pos = pos.rename(columns={'date': 'fecha'})
pos = pos.merge(top_celdas[['lat_era5', 'lon_era5']], on=['lat_era5', 'lon_era5'], how='inner')

neg = negat[['lat_era5', 'lon_era5', 't2m_max', 't2m_mean', 't2m_anomalia',
             'wind_max', 'wind_dir', 'precip', 'dias_sin_lluvia', 'sp_min',
             'sp_delta', 'swvl1', 'ndvi', 'ndvi_anomalia', 'fecha']].copy()
neg['fuego'] = 0
neg = neg.merge(top_celdas[['lat_era5', 'lon_era5']], on=['lat_era5', 'lon_era5'], how='inner')

df = pd.concat([pos, neg], ignore_index=True)
df['fecha'] = pd.to_datetime(df['fecha'])
df.drop(columns=['sp_delta'], inplace=True, errors='ignore')
print(f"Dataset base (solo top celdas): {df.shape}")

# -------------------- 4. Cargar datos de Chile y cordillera --------------------
print("Cargando datos de Chile y cordillera...")
chile_data = pd.read_parquet(ERA5_DIR / "datos_chile.parquet")
chile_data['fecha'] = pd.to_datetime(chile_data['fecha'])
df = df.merge(chile_data, on='fecha', how='left')
print(f"Después de merge con Chile: {df.shape}")

cord_data = pd.read_parquet(ERA5_DIR / "presion_cordillera.parquet")
cord_data['fecha'] = pd.to_datetime(cord_data['fecha'])
df = df.merge(cord_data, on='fecha', how='left')
print(f"Después de merge con cordillera: {df.shape}")

df['pressure_gradient'] = (df['presion_chile_hpa'] * 100) - df['sp_min']
df['pressure_gradient'] = df['pressure_gradient'].fillna(0)
df['pressure_gradient_zonda'] = (df['presion_chile_hpa'] - df['presion_cordillera_hpa']) * 100
df['pressure_gradient_zonda'] = df['pressure_gradient_zonda'].fillna(0)
df['hr_min_chile'] = df['hr_min_chile'].fillna(df['hr_min_chile'].median())
df['wind_gust_max_chile'] = df['wind_gust_max_chile'].fillna(0)

alt_celdas = pd.read_parquet(ERA5_DIR / "altitud_celdas.parquet")
df = df.merge(alt_celdas, on=['lat_era5', 'lon_era5'], how='left')
df['altitud'] = df['altitud'].fillna(df['altitud'].median())
print("Años en df:", df['fecha'].dt.year.min(), "a", df['fecha'].dt.year.max())

# -------------------- 5. Crear columnas temporales básicas (mes, dia_año) antes de dividir --------------------
print("Creando columnas temporales básicas...")
df['dia_año'] = df['fecha'].dt.dayofyear
df['sin_dia'] = np.sin(2 * np.pi * df['dia_año'] / 365.25)
df['cos_dia'] = np.cos(2 * np.pi * df['dia_año'] / 365.25)
df['mes'] = df['fecha'].dt.month

# -------------------- 6. División temporal --------------------
print("Dividiendo en train/val/test...")
df = df.sort_values(['lat_era5', 'lon_era5', 'fecha'])
train_df = df[df['fecha'].dt.year <= 2016].copy()
val_df   = df[(df['fecha'].dt.year >= 2017) & (df['fecha'].dt.year <= 2018)].copy()
test_df  = df[df['fecha'].dt.year >= 2019].copy()
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# -------------------- 7. Calcular climatologías SOLO con train --------------------
print("Calculando climatologías a partir de train...")
clim_t2m = train_df.groupby(['lat_era5', 'lon_era5', 'mes'])['t2m_mean'].mean().reset_index()
clim_t2m.rename(columns={'t2m_mean': 't2m_clim'}, inplace=True)
clim_ndvi = train_df.groupby(['lat_era5', 'lon_era5', 'mes'])['ndvi'].mean().reset_index()
clim_ndvi.rename(columns={'ndvi': 'ndvi_clim'}, inplace=True)

# Guardar climatologías
clim_t2m.to_parquet(ERA5_DIR / "climatologia_t2m.parquet", index=False)
clim_ndvi.to_parquet(ERA5_DIR / "climatologia_ndvi.parquet", index=False)
print("✅ Climatologías guardadas")

# -------------------- 8. Función para aplicar features (recibe climatologías) --------------------
def apply_features(df_chunk, clim_t2m, clim_ndvi):
    df_chunk = df_chunk.copy()
    # Climatología t2m
    df_chunk = df_chunk.merge(clim_t2m, on=['lat_era5', 'lon_era5', 'mes'], how='left')
    df_chunk['t2m_anomalia_aprox'] = df_chunk['t2m_mean'] - df_chunk['t2m_clim']
    # NDVI lag 15
    df_chunk = df_chunk.sort_values(['lat_era5', 'lon_era5', 'fecha'])
    df_chunk['ndvi_lag15'] = df_chunk.groupby(['lat_era5', 'lon_era5'])['ndvi'].shift(15)
    df_chunk['ndvi_lag15'] = df_chunk.groupby(['lat_era5', 'lon_era5'])['ndvi_lag15'].ffill()
    df_chunk['ndvi_lag15'] = df_chunk['ndvi_lag15'].fillna(df_chunk['ndvi'].mean())
    # Climatología NDVI
    df_chunk = df_chunk.merge(clim_ndvi, on=['lat_era5', 'lon_era5', 'mes'], how='left')
    df_chunk['ndvi_anomalia_aprox'] = df_chunk['ndvi_lag15'] - df_chunk['ndvi_clim']
    # Balance hídrico
    df_chunk['precip_7d'] = df_chunk.groupby(['lat_era5', 'lon_era5'])['precip'].transform(
        lambda x: x.rolling(7, min_periods=1).sum()
    )
    df_chunk['t2m_max_7d'] = df_chunk.groupby(['lat_era5', 'lon_era5'])['t2m_max'].transform(
        lambda x: x.rolling(7, min_periods=1).mean()
    )
    df_chunk['t2m_mean_7d'] = df_chunk.groupby(['lat_era5', 'lon_era5'])['t2m_mean'].transform(
        lambda x: x.rolling(7, min_periods=1).mean()
    )
    et_7d = 0.0023 * (df_chunk['t2m_mean_7d'] + 17.8) * (df_chunk['t2m_max_7d'] - df_chunk['t2m_mean_7d']).clip(lower=0) * 7
    df_chunk['balance_hidrico_7d'] = df_chunk['precip_7d'] - et_7d
    df_chunk['swvl1_aprox'] = np.clip(0.1 + 0.005 * df_chunk['balance_hidrico_7d'], 0.05, 0.35)
    return df_chunk

# -------------------- 9. Aplicar features a train, val, test --------------------
print("Aplicando features a train/val/test...")
train_df = apply_features(train_df, clim_t2m, clim_ndvi)
val_df   = apply_features(val_df,   clim_t2m, clim_ndvi)
test_df  = apply_features(test_df,  clim_t2m, clim_ndvi)


# -------------------- 10. Tablas estáticas --------------------
print("Integrando tablas estáticas...")

# Densidad poblacional
dens_pob = pd.read_parquet(ERA5_DIR / "densidad_poblacional_celdas.parquet")
train_df = train_df.merge(dens_pob, on=['lat_era5', 'lon_era5'], how='left')
val_df   = val_df.merge(dens_pob, on=['lat_era5', 'lon_era5'], how='left')
test_df  = test_df.merge(dens_pob, on=['lat_era5', 'lon_era5'], how='left')
train_df['densidad_poblacional'] = train_df['densidad_poblacional'].fillna(0)
val_df['densidad_poblacional']   = val_df['densidad_poblacional'].fillna(0)
test_df['densidad_poblacional']  = test_df['densidad_poblacional'].fillna(0)

# Rayos
rayos_df = pd.read_parquet(ERA5_DIR / "rayos_celdas.parquet")
train_df = train_df.merge(rayos_df, on=['lat_era5', 'lon_era5', 'dia_año'], how='left')
val_df   = val_df.merge(rayos_df, on=['lat_era5', 'lon_era5', 'dia_año'], how='left')
test_df  = test_df.merge(rayos_df, on=['lat_era5', 'lon_era5', 'dia_año'], how='left')
train_df['tasa_rayos'] = train_df['tasa_rayos'].fillna(0)
val_df['tasa_rayos']   = val_df['tasa_rayos'].fillna(0)
test_df['tasa_rayos']  = test_df['tasa_rayos'].fillna(0)

# Feriados (cache)
feriados_cache = ERA5_DIR / "feriados.parquet"
if not feriados_cache.exists():
    all_holidays = []
    for year in range(2000, 2025):
        all_holidays.extend(obtener_feriados_argentina(year))
    feriados_df = pd.DataFrame({'fecha': all_holidays})
    feriados_df['fecha'] = pd.to_datetime(feriados_df['fecha'])
    feriados_df.to_parquet(feriados_cache, index=False)
feriados_df = pd.read_parquet(feriados_cache)
feriados_set = set(feriados_df['fecha'].dt.strftime('%Y-%m-%d'))
train_df['es_feriado'] = train_df['fecha'].dt.strftime('%Y-%m-%d').isin(feriados_set).astype(int)
val_df['es_feriado']   = val_df['fecha'].dt.strftime('%Y-%m-%d').isin(feriados_set).astype(int)
test_df['es_feriado']  = test_df['fecha'].dt.strftime('%Y-%m-%d').isin(feriados_set).astype(int)
train_df['es_finde_semana'] = (train_df['fecha'].dt.dayofweek >= 5).astype(int)
val_df['es_finde_semana']   = (val_df['fecha'].dt.dayofweek >= 5).astype(int)
test_df['es_finde_semana']  = (test_df['fecha'].dt.dayofweek >= 5).astype(int)

# Humedad relativa
hr_hist = pd.read_parquet(ERA5_DIR / "hr_historico.parquet")
hr_hist['fecha'] = pd.to_datetime(hr_hist['fecha'])
train_df = train_df.merge(hr_hist, on=['lat_era5', 'lon_era5', 'fecha'], how='left')
val_df   = val_df.merge(hr_hist, on=['lat_era5', 'lon_era5', 'fecha'], how='left')
test_df  = test_df.merge(hr_hist, on=['lat_era5', 'lon_era5', 'fecha'], how='left')
train_df['hr_mean'] = train_df['hr_mean'].fillna(train_df['hr_mean'].median())
val_df['hr_mean']   = val_df['hr_mean'].fillna(val_df['hr_mean'].median())
test_df['hr_mean']  = test_df['hr_mean'].fillna(test_df['hr_mean'].median())
train_df['temp_hr'] = train_df['t2m_max'] * (100 - train_df['hr_mean']) / 100
val_df['temp_hr']   = val_df['t2m_max'] * (100 - val_df['hr_mean']) / 100
test_df['temp_hr']  = test_df['t2m_max'] * (100 - test_df['hr_mean']) / 100

# FWI
fwi_hist = pd.read_parquet(ERA5_DIR / "fwi_historico.parquet")
fwi_hist['fecha'] = pd.to_datetime(fwi_hist['fecha'])
train_df = train_df.merge(fwi_hist[['lat_era5', 'lon_era5', 'fecha', 'fwi', 'isi', 'bui']],
                          on=['lat_era5', 'lon_era5', 'fecha'], how='left')
val_df   = val_df.merge(fwi_hist[['lat_era5', 'lon_era5', 'fecha', 'fwi', 'isi', 'bui']],
                        on=['lat_era5', 'lon_era5', 'fecha'], how='left')
test_df  = test_df.merge(fwi_hist[['lat_era5', 'lon_era5', 'fecha', 'fwi', 'isi', 'bui']],
                         on=['lat_era5', 'lon_era5', 'fecha'], how='left')
train_df['fwi'] = train_df['fwi'].fillna(0)
val_df['fwi']   = val_df['fwi'].fillna(0)
test_df['fwi']  = test_df['fwi'].fillna(0)
train_df['isi'] = train_df['isi'].fillna(0)
val_df['isi']   = val_df['isi'].fillna(0)
test_df['isi']  = test_df['isi'].fillna(0)
train_df['bui'] = train_df['bui'].fillna(0)
val_df['bui']   = val_df['bui'].fillna(0)
test_df['bui']  = test_df['bui'].fillna(0)

# -------------------- 11. Enriquecimiento de negativos (solo train) --------------------
if ENRIQUECER:
    print("Enriqueciendo muestras negativas en train...")
    mask_neg = train_df['fuego'] == 0
    mask_extremo = (
        (train_df.loc[mask_neg, 't2m_max'] > CONDICIONES_EXTREMAS['t2m_max']) &
        (train_df.loc[mask_neg, 'dias_sin_lluvia'] > CONDICIONES_EXTREMAS['dias_sin_lluvia']) &
        (train_df.loc[mask_neg, 'wind_max'] > CONDICIONES_EXTREMAS['wind_max']) &
        (train_df.loc[mask_neg, 'swvl1_aprox'] < CONDICIONES_EXTREMAS['swvl1_aprox'])
    )
    neg_extremos = train_df[mask_neg][mask_extremo].copy()
    print(f"  Negativos originales en train: {mask_neg.sum()}")
    print(f"  Negativos con condiciones extremas sin fuego: {len(neg_extremos)}")
    if len(neg_extremos) > 0:
        neg_extremos_dup = pd.concat([neg_extremos] * FACTOR_DUPLICACION, ignore_index=True)
        train_df = pd.concat([train_df, neg_extremos_dup], ignore_index=True)
        print(f"  Train después de duplicar extremos: {len(train_df)}")
    else:
        print("  ⚠️ No se encontraron negativos con condiciones extremas.")

# -------------------- 12. Features avanzadas --------------------
print("Creando features avanzadas...")
dfs = {'train_df': train_df, 'val_df': val_df, 'test_df': test_df}
for key, df_chunk in dfs.items():
    df_chunk = df_chunk.sort_values(['lat_era5', 'lon_era5', 'fecha'])
    df_chunk['t2m_max_3d_avg'] = df_chunk.groupby(['lat_era5', 'lon_era5'])['t2m_max'].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
    df_chunk['precip_3d_sum'] = df_chunk.groupby(['lat_era5', 'lon_era5'])['precip'].transform(
        lambda x: x.rolling(3, min_periods=1).sum()
    )
    df_chunk['rayos_ndvi'] = df_chunk['tasa_rayos'] * df_chunk['ndvi_anomalia_aprox'].clip(lower=0)
    df_chunk['temp_ndvi'] = df_chunk['t2m_anomalia_aprox'] * df_chunk['ndvi_anomalia_aprox'].clip(lower=0)
    df_chunk['fire_danger'] = (df_chunk['t2m_max'] - 20).clip(lower=0) * (df_chunk['wind_max'] / 10) * (1 - df_chunk['swvl1_aprox'])
    dfs[key] = df_chunk
train_df, val_df, test_df = dfs['train_df'], dfs['val_df'], dfs['test_df']

# -------------------- 13. Fuegos activos lag 1 --------------------
print("Integrando fuegos_activos_lag1...")
fuegos_hist = pd.read_parquet(ERA5_DIR / "fuegos_activos_historicos.parquet")
fuegos_hist['fecha'] = pd.to_datetime(fuegos_hist['fecha'])
dfs = {'train_df': train_df, 'val_df': val_df, 'test_df': test_df}
for key, df_chunk in dfs.items():
    df_chunk = df_chunk.merge(fuegos_hist, on=['lat_era5', 'lon_era5', 'fecha'], how='left')
    df_chunk['fuegos_activos'] = df_chunk['fuegos_activos'].fillna(0).astype(int)
    df_chunk = df_chunk.sort_values(['lat_era5', 'lon_era5', 'fecha'])
    df_chunk['fuegos_activos_lag1'] = df_chunk.groupby(['lat_era5', 'lon_era5'])['fuegos_activos'].shift(1).fillna(0).astype(int)
    df_chunk.drop(columns=['fuegos_activos'], inplace=True)
    dfs[key] = df_chunk
train_df, val_df, test_df = dfs['train_df'], dfs['val_df'], dfs['test_df']

# -------------------- 14. Features finales --------------------
FULL_FEATURES = [
    't2m_max', 't2m_mean', 't2m_anomalia_aprox', 'wind_max', 'wind_dir',
    'precip', 'dias_sin_lluvia', 'sp_min', 'swvl1_aprox',
    'ndvi_lag15', 'ndvi_anomalia_aprox', 'sin_dia', 'cos_dia',
    'densidad_poblacional', 'es_feriado', 'es_finde_semana', 'tasa_rayos',
    't2m_max_3d_avg', 'precip_3d_sum', 'rayos_ndvi', 'temp_ndvi', 'fire_danger',
    'fuegos_activos_lag1', 'hr_mean', 'temp_hr', 'fwi', 'isi', 'bui',
    'pressure_gradient', 'hr_min_chile', 'wind_gust_max_chile',
    'pressure_gradient_zonda', 'altitud'
]
print(f"Features totales: {len(FULL_FEATURES)}")

# Eliminar filas con NaN
train_df = train_df.dropna(subset=FULL_FEATURES)
val_df   = val_df.dropna(subset=FULL_FEATURES)
test_df  = test_df.dropna(subset=FULL_FEATURES)

X_train, y_train = train_df[FULL_FEATURES], train_df['fuego']
X_val,   y_val   = val_df[FULL_FEATURES],   val_df['fuego']
X_test,  y_test  = test_df[FULL_FEATURES],  test_df['fuego']

print(f"\nTrain (≤2016): {X_train.shape} | Pos={y_train.sum()} | Neg={len(y_train)-y_train.sum()}")
print(f"Val  (2017-18): {X_val.shape}   | Pos={y_val.sum()}   | Neg={len(y_val)-y_val.sum()}")
print(f"Test (≥2019):  {X_test.shape}   | Pos={y_test.sum()}  | Neg={len(y_test)-y_test.sum()}")

# -------------------- 15. Entrenamiento --------------------
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f"\nEntrenando XGBoost con scale_pos_weight={scale_pos_weight:.2f}...")

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    eval_metric='auc',
    use_label_encoder=False,
    tree_method='hist',
    early_stopping_rounds=30,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

# -------------------- 16. Evaluación --------------------
y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
umbral_medio = top_celdas['umbral_celda'].mean()
y_pred = (y_prob >= umbral_medio).astype(int)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f"\n{'='*50}")
print(f"AUC-ROC:                      {auc:.4f}")
print(f"PR-AUC:                       {pr_auc:.4f}")
print(f"Umbral medio ({umbral_medio:.3f}) → Precisión: {prec:.3f} | Recall: {rec:.3f}")
print(f"{'='*50}")

feat_imp = pd.Series(model.feature_importances_, index=FULL_FEATURES)
print("\nTop 10 features por importancia:")
print(feat_imp.sort_values(ascending=False).head(10))



# ==================== CALIBRACIÓN MANUAL (Platt scaling) ====================
print("\nCalibrando probabilidades con regresión logística...")
from sklearn.linear_model import LogisticRegression

# Predicciones del modelo original sobre validación
y_val_prob = model.predict_proba(X_val)[:, 1].reshape(-1, 1)
# Entrenar calibrador
calibrator = LogisticRegression(random_state=RANDOM_STATE)
calibrator.fit(y_val_prob, y_val)

# Aplicar a test
y_prob_calib = calibrator.predict_proba(model.predict_proba(X_test)[:, 1].reshape(-1, 1))[:, 1]

auc_calib = roc_auc_score(y_test, y_prob_calib)
pr_auc_calib = average_precision_score(y_test, y_prob_calib)

print(f"AUC-ROC calibrado: {auc_calib:.4f} (original: {auc:.4f})")
print(f"PR-AUC calibrado:  {pr_auc_calib:.4f} (original: {pr_auc:.4f})")
print("\nComparación de probabilidades medias:")
print(f"  Para aciertos (VP): original={y_prob[y_test==1].mean():.3f}, calibrado={y_prob_calib[y_test==1].mean():.3f}")
print(f"  Para falsas alarmas (FP): original={y_prob[y_test==0].mean():.3f}, calibrado={y_prob_calib[y_test==0].mean():.3f}")

# -------------------- 17. Guardar modelo --------------------
model_data = {
    'modelo':       model,
    'features':     FULL_FEATURES,
    'auc_test':     auc,
    'pr_auc_test':  pr_auc,
}
with open(ERA5_DIR / "modelo_xgb_mejorado.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("\n✅ Artefactos guardados:")
print("   - modelo_xgb_mejorado.pkl")
print("   - top_celdas.parquet")
print("   - umbrales_celda.parquet")
print("   - altitud_celdas.parquet")
print("   - climatologia_t2m.parquet")
print("   - climatologia_ndvi.parquet")

## Backtesting

In [ ]:
"""
Backtesting del modelo XGBoost (top 50 celdas dentro de Salta).
Versión corregida: incluye todas las features del modelo final
(pressure_gradient_zonda, altitud, etc.).
"""

import pandas as pd
import numpy as np
import pickle
import requests
from pathlib import Path
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, average_precision_score

ERA5_DIR = Path("era5_salta")
YEAR_TEST = 2023   # Cambia a 2019 para todo el período

# Función para obtener feriados
def obtener_feriados_argentina(anio):
    url = f"https://date.nager.at/api/v3/PublicHolidays/{anio}/AR"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return [h['date'] for h in response.json()]
    except Exception as e:
        print(f"⚠️ Error en feriados {anio}: {e}")
        return []

# Cargar modelo y top celdas
print("Cargando modelo y artefactos...")
with open(ERA5_DIR / "modelo_xgb_mejorado.pkl", "rb") as f:
    model_data = pickle.load(f)
model = model_data['modelo']
features = model_data['features']

top_celdas = pd.read_parquet(ERA5_DIR / "top_celdas.parquet")
celdas_monitoreadas = top_celdas[['lat_era5', 'lon_era5']]

# Cargar altitud de celdas
alt_celdas = pd.read_parquet(ERA5_DIR / "altitud_celdas.parquet")
alt_dict = alt_celdas.set_index(['lat_era5', 'lon_era5'])['altitud'].to_dict()

# Cargar datos base
print("Cargando datos base...")
fired = pd.read_parquet(ERA5_DIR / "fired_completo.parquet")
negat = pd.read_parquet(ERA5_DIR / "negativos.parquet")

# Filtrar solo top celdas
pos = fired[['lat_era5', 'lon_era5', 't2m_max', 't2m_mean', 't2m_anomalia',
             'wind_max', 'wind_dir', 'precip', 'dias_sin_lluvia', 'sp_min',
             'sp_delta', 'swvl1', 'ndvi', 'ndvi_anomalia', 'date']].copy()
pos['fuego'] = 1
pos = pos.rename(columns={'date': 'fecha'})
pos = pos.merge(celdas_monitoreadas, on=['lat_era5', 'lon_era5'], how='inner')

neg = negat[['lat_era5', 'lon_era5', 't2m_max', 't2m_mean', 't2m_anomalia',
             'wind_max', 'wind_dir', 'precip', 'dias_sin_lluvia', 'sp_min',
             'sp_delta', 'swvl1', 'ndvi', 'ndvi_anomalia', 'fecha']].copy()
neg['fuego'] = 0
neg = neg.merge(celdas_monitoreadas, on=['lat_era5', 'lon_era5'], how='inner')

df = pd.concat([pos, neg], ignore_index=True)
df['fecha'] = pd.to_datetime(df['fecha'])
df.drop(columns=['sp_delta'], inplace=True, errors='ignore')
print(f"Dataset base (solo top celdas): {df.shape}")

# ==================== 1. Filtrar por fecha ANTES de calcular features ====================
if YEAR_TEST == 2019:
    df = df[df['fecha'].dt.year >= 2019].copy()
else:
    df = df[df['fecha'].dt.year == YEAR_TEST].copy()
print(f"Período de test (sin features aún): {df['fecha'].min().date()} a {df['fecha'].max().date()}")

# ==================== 2. Cargar datos de Chile y cordillera ====================
print("Cargando datos de Chile...")
chile_data = pd.read_parquet(ERA5_DIR / "datos_chile.parquet")
df = df.merge(chile_data, on='fecha', how='left')
df['pressure_gradient'] = (df['presion_chile_hpa'] * 100) - df['sp_min']
df['pressure_gradient'] = df['pressure_gradient'].fillna(0)
df['hr_min_chile'] = df['hr_min_chile'].fillna(df['hr_min_chile'].median())
df['wind_gust_max_chile'] = df['wind_gust_max_chile'].fillna(0)

print("Cargando datos de cordillera...")
cord_data = pd.read_parquet(ERA5_DIR / "presion_cordillera.parquet")
df = df.merge(cord_data, on='fecha', how='left')
df['presion_cordillera_hpa'] = df['presion_cordillera_hpa'].fillna(0)
df['pressure_gradient_zonda'] = (df['presion_chile_hpa'] - df['presion_cordillera_hpa']) * 100
df['pressure_gradient_zonda'] = df['pressure_gradient_zonda'].fillna(0)

# Altitud
df['altitud'] = df.apply(lambda row: alt_dict.get((row['lat_era5'], row['lon_era5']), 1000), axis=1)

# ==================== 3. Calcular todas las features ====================
print("Calculando features...")
df['dia_año'] = df['fecha'].dt.dayofyear
df['sin_dia'] = np.sin(2 * np.pi * df['dia_año'] / 365.25)
df['cos_dia'] = np.cos(2 * np.pi * df['dia_año'] / 365.25)
df['mes'] = df['fecha'].dt.month

# Climatología t2m
clim_t2m = pd.read_parquet(ERA5_DIR / "climatologia_t2m.parquet")
df = df.merge(clim_t2m, on=['lat_era5', 'lon_era5', 'mes'], how='left')
df['t2m_anomalia_aprox'] = df['t2m_mean'] - df['t2m_clim']

# NDVI lag 15 (solo ffill)
df = df.sort_values(['lat_era5', 'lon_era5', 'fecha'])
df['ndvi_lag15'] = df.groupby(['lat_era5', 'lon_era5'])['ndvi'].shift(15)
df['ndvi_lag15'] = df.groupby(['lat_era5', 'lon_era5'])['ndvi_lag15'].ffill()
df['ndvi_lag15'] = df['ndvi_lag15'].fillna(df['ndvi'].mean())

# Climatología NDVI
clim_ndvi = pd.read_parquet(ERA5_DIR / "climatologia_ndvi.parquet")
df = df.merge(clim_ndvi, on=['lat_era5', 'lon_era5', 'mes'], how='left')
df['ndvi_anomalia_aprox'] = df['ndvi_lag15'] - df['ndvi_clim']

# Balance hídrico
df['precip_7d'] = df.groupby(['lat_era5', 'lon_era5'])['precip'].transform(
    lambda x: x.rolling(7, min_periods=1).sum()
)
df['t2m_max_7d'] = df.groupby(['lat_era5', 'lon_era5'])['t2m_max'].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
df['t2m_mean_7d'] = df.groupby(['lat_era5', 'lon_era5'])['t2m_mean'].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
et_7d = 0.0023 * (df['t2m_mean_7d'] + 17.8) * (df['t2m_max_7d'] - df['t2m_mean_7d']).clip(lower=0) * 7
df['balance_hidrico_7d'] = df['precip_7d'] - et_7d
df['swvl1_aprox'] = np.clip(0.1 + 0.005 * df['balance_hidrico_7d'], 0.05, 0.35)

# Densidad poblacional
dens_pob = pd.read_parquet(ERA5_DIR / "densidad_poblacional_celdas.parquet")
df = df.merge(dens_pob, on=['lat_era5', 'lon_era5'], how='left')
df['densidad_poblacional'] = df['densidad_poblacional'].fillna(0)

# Rayos
rayos_df = pd.read_parquet(ERA5_DIR / "rayos_celdas.parquet")
df = df.merge(rayos_df, on=['lat_era5', 'lon_era5', 'dia_año'], how='left')
df['tasa_rayos'] = df['tasa_rayos'].fillna(0)

# Feriados
years = df['fecha'].dt.year.unique()
all_holidays = []
for year in years:
    all_holidays.extend(obtener_feriados_argentina(year))
holidays_set = set(all_holidays)
df['es_feriado'] = df['fecha'].dt.strftime('%Y-%m-%d').isin(holidays_set).astype(int)
df['es_finde_semana'] = (df['fecha'].dt.dayofweek >= 5).astype(int)

# Features avanzadas
df = df.sort_values(['lat_era5', 'lon_era5', 'fecha'])
df['t2m_max_3d_avg'] = df.groupby(['lat_era5', 'lon_era5'])['t2m_max'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
df['precip_3d_sum'] = df.groupby(['lat_era5', 'lon_era5'])['precip'].transform(
    lambda x: x.rolling(3, min_periods=1).sum()
)
df['rayos_ndvi'] = df['tasa_rayos'] * df['ndvi_anomalia_aprox'].clip(lower=0)
df['temp_ndvi'] = df['t2m_anomalia_aprox'] * df['ndvi_anomalia_aprox'].clip(lower=0)
df['fire_danger'] = (df['t2m_max'] - 20).clip(lower=0) * (df['wind_max'] / 10) * (1 - df['swvl1_aprox'])

# Fuegos activos históricos con lag 1
fuegos_hist = pd.read_parquet(ERA5_DIR / "fuegos_activos_historicos.parquet")
fuegos_hist = fuegos_hist.merge(celdas_monitoreadas, on=['lat_era5', 'lon_era5'], how='inner')
df = df.merge(fuegos_hist, on=['lat_era5', 'lon_era5', 'fecha'], how='left')
df['fuegos_activos'] = df['fuegos_activos'].fillna(0).astype(int)
df = df.sort_values(['lat_era5', 'lon_era5', 'fecha'])
df['fuegos_activos_lag1'] = df.groupby(['lat_era5', 'lon_era5'])['fuegos_activos'].shift(1).fillna(0).astype(int)

# Humedad relativa
hr_hist = pd.read_parquet(ERA5_DIR / "hr_historico.parquet")
hr_hist['fecha'] = pd.to_datetime(hr_hist['fecha'])
df = df.merge(hr_hist, on=['lat_era5', 'lon_era5', 'fecha'], how='left')
df['hr_mean'] = df['hr_mean'].fillna(df['hr_mean'].median())
df['temp_hr'] = df['t2m_max'] * (100 - df['hr_mean']) / 100

# FWI
fwi_hist = pd.read_parquet(ERA5_DIR / "fwi_historico.parquet")
fwi_hist['fecha'] = pd.to_datetime(fwi_hist['fecha'])
df = df.merge(fwi_hist[['lat_era5', 'lon_era5', 'fecha', 'fwi', 'isi', 'bui']],
              on=['lat_era5', 'lon_era5', 'fecha'], how='left')
df['fwi'] = df['fwi'].fillna(0)
df['isi'] = df['isi'].fillna(0)
df['bui'] = df['bui'].fillna(0)

# ==================== 4. Preparar X_test y y_test ====================
missing = set(features) - set(df.columns)
if missing:
    print(f"⚠️ Faltan columnas: {missing}")
    for col in missing:
        df[col] = 0

X_test = df[features]
y_test = df['fuego']
print(f"Registros finales en test: {len(X_test)} | Positivos: {y_test.sum()}")

# ==================== 5. Predicción y evaluación ====================
y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

UMBRAL_FIJO = 0.35
y_pred_fijo = (y_prob >= UMBRAL_FIJO).astype(int)
prec_fijo = precision_score(y_test, y_pred_fijo, zero_division=0)
rec_fijo = recall_score(y_test, y_pred_fijo, zero_division=0)
f1_fijo = f1_score(y_test, y_pred_fijo, zero_division=0)

# Umbrales por celda
umbrales_celda = pd.read_parquet(ERA5_DIR / "umbrales_celda.parquet")
umbral_dict = {(row['lat_era5'], row['lon_era5']): row['umbral_celda'] for _, row in umbrales_celda.iterrows()}
df['umbral'] = df.apply(lambda row: umbral_dict.get((row['lat_era5'], row['lon_era5']), 0.35), axis=1)
y_pred_celda = (y_prob >= df['umbral']).astype(int)
prec_celda = precision_score(y_test, y_pred_celda, zero_division=0)
rec_celda = recall_score(y_test, y_pred_celda, zero_division=0)
f1_celda = f1_score(y_test, y_pred_celda, zero_division=0)

print(f"\n{'='*50}")
print(f"AUC-ROC: {auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"Umbral fijo {UMBRAL_FIJO} -> Precisión: {prec_fijo:.3f}, Recall: {rec_fijo:.3f}, F1: {f1_fijo:.3f}")
print(f"Umbrales por celda -> Precisión: {prec_celda:.3f}, Recall: {rec_celda:.3f}, F1: {f1_celda:.3f}")
print(f"{'='*50}")


# ==================== Probar múltiples umbrales ====================
print("\n" + "="*60)
print("EVALUACIÓN CON DIFERENTES UMBRALES")
print("="*60)

umbrales = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
resultados_umbrales = []

for umbral in umbrales:
    y_pred = (y_prob >= umbral).astype(int)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    alertas = y_pred.sum()
    aciertos = (y_pred & y_test).sum()
    print(f"Umbral {umbral:.2f}: Precisión={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}, Alertas={alertas}, Aciertos={aciertos}")
    resultados_umbrales.append({
        'umbral': umbral,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'alertas': alertas,
        'aciertos': aciertos
    })

# Mostrar resumen
df_umbrales = pd.DataFrame(resultados_umbrales)
print("\n" + "="*60)
print("RESUMEN:")
print(df_umbrales.to_string(index=False))

## 🚀 Production Pipeline & Automated Alerting

This section runs the operational wildfire monitoring system.

Using the trained model, the pipeline collects current environmental and meteorological data, generates wildfire risk assessments, updates geospatial visualizations and automatically distributes alerts through Telegram.

Core functions include:

- Environmental data ingestion
- Risk prediction and scoring
- Geospatial risk mapping
- Automated alert generation
- Telegram-based notification workflows
- Continuous operational monitoring

The objective is to transform predictive analytics into actionable intelligence for early detection and rapid response.

## Production

🎯 **Which threshold to choose?**  
It depends on the client’s priorities:

- **If the priority is to detect as many fires as possible (high recall):** choose 0.35 (recall 89.4%, precision 15.9%).

- **If the priority is to reduce false alarms (high precision):** choose 0.55 (precision 25.3%, recall 56.3%). The F1 score is highest at 0.55 (0.349).

- **Balance between both (maximum F1):** 0.55 gives the best F1, but recall drops to 56%. For a higher recall with better precision than 0.35, you could choose 0.50 (precision 21.7%, recall 72.4%, F1 0.333).

- **For a marketable product:**  
  - If the client is an authority with resources to handle false alarms (firefighters, civil defence) → threshold 0.35 (no fires missed).  
  - If the client has limited resources (forestry company, small municipality) → threshold 0.50 or 0.55 (fewer false alarms, but ~30‑40% of fires are missed).

You can present this table to the client and let them choose according to their tolerance.

In [ ]:
#!/usr/bin/env python3
"""
Sistema de alerta temprana de incendios forestales - Salta.
- Monitorea top 50 celdas con mayor frecuencia histórica.
- Usa umbral fijo 0.50 (modificable, ver Backtesting).
- Predice riesgo para día siguiente usando pronóstico + lluvia real últimos 7 días.
- Incorpora humedad relativa, interacción temp_hr, FWI, gradientes de presión, altitud.
- Mapa: alertas con ícono fuego, top 5 numerado.
- Leyenda del mapa: top 20 alertas con departamento y probabilidad.
- Mensaje Telegram: top 20 (top 5 con explicación detallada).
- Guarda historial de alertas.
"""

import requests
import pandas as pd
import numpy as np
import pickle
import folium
import geopandas as gpd
import time  
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from datetime import datetime, timedelta
from pathlib import Path


# ========================= CONFIGURACIÓN =========================
ERA5_DIR        = Path("era5_salta")
TELEGRAM_TOKEN  = #"Insert your own token here"
TELEGRAM_CHAT_ID = #"Insert your chat ID here"
FIRMS_MAP_KEY   = #"Insert your key here"
UMBRAL = 0.50                                                       # Umbral fijo (ajustable)
# =================================================================

# --- Funciones auxiliares ---
def cargar_limite_salta():
    limite_path = ERA5_DIR / "salta_provincia.gpkg"
    if not limite_path.exists():
        print("Descargando límite de Salta desde OpenStreetMap...")
        import osmnx as ox
        salta = ox.geocode_to_gdf("Provincia de Salta, Argentina")
        salta = salta.to_crs("EPSG:4326")
        salta.to_file(limite_path, driver='GPKG')
        print("✅ Límite de Salta (OSM) guardado.")
    return gpd.read_file(limite_path)

def filtrar_en_salta(df, limite_salta):
    if df.empty:
        return df
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=[Point(lon, lat) for lat, lon in zip(df['lat'], df['lon'])],
        crs="EPSG:4326"
    )
    dentro = gpd.sjoin(gdf, limite_salta[['geometry']], how='inner', predicate='within')
    return df.loc[dentro.index].copy()

def cargar_modelo():
    with open(ERA5_DIR / "modelo_xgb_mejorado.pkl", "rb") as f:
        return pickle.load(f)

def obtener_pronostico_dia_objetivo(lat, lon, fecha):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_mean",
            "relative_humidity_2m_mean",
            "relative_humidity_2m_min",      
            "wind_speed_10m_max",
            "wind_gusts_10m_max",            
            "wind_direction_10m_dominant",
            "precipitation_sum",
            "surface_pressure_min"
        ],
        "timezone":   "America/Argentina/Salta",
        "start_date": fecha.strftime("%Y-%m-%d"),
        "end_date":   fecha.strftime("%Y-%m-%d")
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()

def obtener_precipitacion_historica(lat, lon, fecha_inicio, fecha_fin):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": fecha_inicio.strftime("%Y-%m-%d"),
        "end_date":   fecha_fin.strftime("%Y-%m-%d"),
        "daily":      "precipitation_sum",
        "timezone":   "America/Argentina/Salta"
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return data['daily']['precipitation_sum']

def obtener_fuegos_activos(bbox_coords):
    url = (
        f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{FIRMS_MAP_KEY}/VIIRS_NOAA20_NRT/"
        f"{bbox_coords[0]},{bbox_coords[1]},{bbox_coords[2]},{bbox_coords[3]}/1"
    )
    try:
        df = pd.read_csv(url)
        print(f"✅ FIRMS: {len(df)} focos activos en las últimas 24h.")
        return df
    except Exception as e:
        print(f"⚠️ Error al descargar FIRMS: {e}. Se asume 0 focos activos.")
        return pd.DataFrame()

def obtener_presion_chile(fecha, retries=3):
    """Obtiene presión mínima diaria en Antofagasta (Chile) con reintentos."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": -23.65,
        "longitude": -70.40,
        "daily": "surface_pressure_min",
        "timezone": "America/Argentina/Salta",
        "start_date": fecha.strftime("%Y-%m-%d"),
        "end_date": fecha.strftime("%Y-%m-%d")
    }
    for i in range(retries):
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json()['daily']['surface_pressure_min'][0]  # hPa
        except Exception as e:
            if i == retries - 1:
                raise
            print(f"  Reintentando presión Chile ({i+1}/{retries})...")
            time.sleep(2)

def obtener_presion_cordillera(fecha, retries=3):
    """Presión en la celda de la cordillera (coordenadas fijas) con reintentos."""
    LAT_CORD = -23.0
    LON_CORD = -64.5
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": LAT_CORD,
        "longitude": LON_CORD,
        "daily": "surface_pressure_min",
        "timezone": "America/Argentina/Salta",
        "start_date": fecha.strftime("%Y-%m-%d"),
        "end_date": fecha.strftime("%Y-%m-%d")
    }
    for i in range(retries):
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json()['daily']['surface_pressure_min'][0]  # hPa
        except Exception as e:
            if i == retries - 1:
                raise
            print(f"  Reintentando presión cordillera ({i+1}/{retries})...")
            time.sleep(2)    
    
def asignar_fuegos_a_celdas(focos_df, celdas_era5):
    if focos_df.empty or 'latitude' not in focos_df.columns:
        return celdas_era5.assign(fuegos_activos_lag1=0)

    coords_celdas = np.radians(celdas_era5[['lat_era5', 'lon_era5']].values)
    coords_focos  = np.radians(focos_df[['latitude', 'longitude']].values)
    tree = BallTree(coords_celdas, metric='haversine')
    _, idx = tree.query(coords_focos, k=1)

    focos_df = focos_df.copy()
    focos_df['lat_era5'] = celdas_era5.iloc[idx.flatten()]['lat_era5'].values
    focos_df['lon_era5'] = celdas_era5.iloc[idx.flatten()]['lon_era5'].values

    conteo = (focos_df.groupby(['lat_era5', 'lon_era5'])
                      .size()
                      .reset_index(name='fuegos_activos_lag1'))

    resultado = celdas_era5.merge(conteo, on=['lat_era5', 'lon_era5'], how='left')
    resultado['fuegos_activos_lag1'] = resultado['fuegos_activos_lag1'].fillna(0).astype(int)
    return resultado

def obtener_departamento(lat, lon, cache={}):
    clave = (round(lat, 2), round(lon, 2))
    if clave in cache:
        return cache[clave]
    url = f"https://apis.datos.gob.ar/georef/api/ubicacion?lat={lat}&lon={lon}"
    try:
        response = requests.get(url, timeout=3)
        data = response.json()
        if 'ubicacion' in data and 'departamento' in data['ubicacion']:
            depto = data['ubicacion']['departamento']['nombre']
            cache[clave] = depto
            return depto
        else:
            cache[clave] = "Desconocido"
            return "Desconocido"
    except Exception:
        cache[clave] = "Error"
        return "Error"

# -------------------- MAPA--------------------
def generar_mapa(alertas_df, limite_salta, fecha_alerta):
    """Mapa: top 5 con círculo numerado rojo, resto ícono fuego azul. Leyenda con top 20."""
    if alertas_df.empty:
        mapa = folium.Map(location=[-24.5, -65.5], zoom_start=7, tiles="CartoDB positron")
        folium.GeoJson(
            limite_salta.to_crs("EPSG:4326").__geo_interface__,
            style_function=lambda _: {'color': '#2c3e50', 'weight': 2, 'dashArray': '5,5'}
        ).add_to(mapa)
        titulo_html = f"""
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
                    z-index: 1000; background-color: white; padding: 10px 20px;
                    border: 2px solid #c0392b; border-radius: 8px;
                    font-family: Arial; font-size: 14px; font-weight: bold;">
            🔥 RIESGO DE INCENDIO — SALTA — {fecha_alerta.strftime('%d/%m/%Y')}
        </div>"""
        mapa.get_root().html.add_child(folium.Element(titulo_html))
        mapa_path = ERA5_DIR / f"mapa_riesgo_{fecha_alerta}.html"
        mapa.save(str(mapa_path))
        print(f"🗺️ Mapa guardado (sin alertas): {mapa_path}")
        return mapa_path

    mapa = folium.Map(location=[-24.5, -65.5], zoom_start=7, tiles="CartoDB positron")
    folium.GeoJson(
        limite_salta.to_crs("EPSG:4326").__geo_interface__,
        style_function=lambda _: {'color': '#2c3e50', 'weight': 2, 'dashArray': '5,5'}
    ).add_to(mapa)

    titulo_html = f"""
    <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
                z-index: 1000; background-color: white; padding: 10px 20px;
                border: 2px solid #c0392b; border-radius: 8px;
                font-family: Arial; font-size: 14px; font-weight: bold;">
        🔥 RIESGO DE INCENDIO — SALTA — {fecha_alerta.strftime('%d/%m/%Y')}
    </div>"""
    mapa.get_root().html.add_child(folium.Element(titulo_html))

    # Ordenar y asignar ranking
    alertas_ordenadas = alertas_df.sort_values('probabilidad', ascending=False).reset_index(drop=True)
    alertas_ordenadas['ranking'] = alertas_ordenadas.index + 1
    top5 = alertas_ordenadas.head(5)
    resto = alertas_ordenadas.iloc[5:]

    # Dibujar top 5 con círculos numerados (rojo)
    for _, row in top5.iterrows():
        num = row['ranking']
        # Color según probabilidad (opcional, pero para mantener consistencia)
        if row['probabilidad'] >= 0.50:
            bg_color = '#e74c3c'
        elif row['probabilidad'] >= 0.40:
            bg_color = '#e67e22'
        else:
            bg_color = '#f1c40f'
        icon = folium.DivIcon(
            html=f"""<div style="background-color:{bg_color}; border-radius:50%; width:28px; height:28px; 
                            display:flex; align-items:center; justify-content:center; font-weight:bold; 
                            color:white; border:2px solid black; font-size:14px;">
                        {num}
                    </div>""",
            icon_size=(28, 28),
            icon_anchor=(14, 14)
        )
        depto = row.get('departamento', obtener_departamento(row['lat'], row['lon']))
        popup_html = f"""
        <div style="font-family:Arial; font-size:13px; min-width:200px">
            <b style="color:{bg_color}">🔥 ALERTA DE INCENDIO (Puesto {num})</b>
            <hr style="margin:5px 0">
            <b>Departamento:</b> {depto}<br>
            <b>Probabilidad:</b> {row['probabilidad']:.1%}<br>
            <b>Coordenadas:</b> {abs(row['lat']):.2f}°S, {abs(row['lon']):.2f}°O<br>
            <b>Factores:</b> {row.get('explicacion', 'Consultar mensaje')}<br>
            <i>Haga clic para más detalles</i>
        </div>"""
        folium.Marker(
            location=[row['lat'], row['lon']],
            popup=folium.Popup(popup_html, max_width=250),
            icon=icon
        ).add_to(mapa)

    # Dibujar el resto con ícono de fuego azul
    for _, row in resto.iterrows():
        icon = folium.Icon(color='blue', icon='fire', prefix='fa')
        depto = row.get('departamento', obtener_departamento(row['lat'], row['lon']))
        popup_html = f"""
        <div style="font-family:Arial; font-size:13px; min-width:200px">
            <b style="color:blue">🔥 ALERTA DE INCENDIO (Puesto {row['ranking']})</b>
            <hr style="margin:5px 0">
            <b>Departamento:</b> {depto}<br>
            <b>Probabilidad:</b> {row['probabilidad']:.1%}<br>
            <b>Coordenadas:</b> {abs(row['lat']):.2f}°S, {abs(row['lon']):.2f}°O<br>
            <b>Factores:</b> {row.get('explicacion', 'Consultar mensaje')}<br>
            <i>Haga clic para más detalles</i>
        </div>"""
        folium.Marker(
            location=[row['lat'], row['lon']],
            popup=folium.Popup(popup_html, max_width=250),
            icon=icon
        ).add_to(mapa)

    # Leyenda con top 20
    top20 = alertas_ordenadas.head(20)
    tabla_html = '<div style="max-height: 400px; overflow-y: auto;"><table style="border-collapse:collapse; width:100%; font-size:12px;">'
    for _, row in top20.iterrows():
        depto = row.get('departamento', obtener_departamento(row['lat'], row['lon']))
        color_fondo = '#e74c3c' if row['ranking'] <= 5 else '#3498db'
        tabla_html += f"""
        <tr>
            <td style="padding:2px 4px; font-weight:bold; color:white; background:{color_fondo}; border-radius:50%; text-align:center; width:24px">{row['ranking']}</td>
            <td style="padding:2px 8px">{depto}</td>
            <td style="padding:2px 4px; font-weight:bold; color:{color_fondo}; text-align:right">{row['probabilidad']:.1%}</td>
        </tr>
        """
    tabla_html += '</table></div>'

    leyenda_html = f"""
    <div style="position: fixed; bottom: 20px; left: 15px; z-index: 1000;
                background-color: white; padding: 14px 16px;
                border: 1px solid #ccc; border-radius: 8px;
                font-family: Arial; font-size: 12px; min-width: 280px;
                max-height: 70vh; overflow-y: auto;
                box-shadow: 2px 2px 6px rgba(0,0,0,0.2);">
        <b style="font-size:13px">🔥 TOP 20 ZONAS CRÍTICAS</b><br><br>
        {tabla_html}
        <hr style="margin:8px 0">
        <span style="color:red;">●</span> <b>Top 5 alertas</b> (numeradas)<br>
        <span style="color:blue;">🔥</span> <b>Otras alertas</b> (puestos 6-20)<br>
        <hr style="margin:8px 0">
        <span style="font-size:11px; color:#777">
            Hacé click en cualquier punto para ver detalles.<br>
            -- Límite provincial (OSM)
        </span>
    </div>"""
    mapa.get_root().html.add_child(folium.Element(leyenda_html))

    mapa_path = ERA5_DIR / f"mapa_riesgo_{fecha_alerta}.html"
    mapa.save(str(mapa_path))
    print(f"🗺️ Mapa guardado: {mapa_path} ({len(alertas_df)} alertas, top 5 numerados)")
    return mapa_path

def enviar_telegram_con_mapa(mensaje_texto, archivo_mapa=None):
    url_text = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    try:
        r = requests.post(url_text,
                          json={"chat_id": TELEGRAM_CHAT_ID,
                                "text": mensaje_texto,
                                "parse_mode": "HTML"},
                          timeout=10)
        r.raise_for_status()
        print("✅ Mensaje enviado a Telegram")
    except Exception as e:
        print(f"❌ Error al enviar mensaje: {e}")

    if archivo_mapa and Path(archivo_mapa).exists():
        url_doc = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendDocument"
        try:
            with open(archivo_mapa, 'rb') as f:
                r = requests.post(
                    url_doc,
                    files={'document': f},
                    data={
                        'chat_id': TELEGRAM_CHAT_ID,
                        'caption': '🗺️ Mapa interactivo — la leyenda muestra el Top 20'
                    },
                    timeout=30
                )
            r.raise_for_status()
            print("✅ Mapa enviado a Telegram")
        except Exception as e:
            print(f"❌ Error al enviar mapa: {e}")

def explicar_riesgo(row):
    factores = []
    # Temperatura
    if row['t2m_max'] > 35:
        factores.append(f"temperatura extremadamente alta ({row['t2m_max']:.0f}°C)")
    elif row['t2m_max'] > 30:
        factores.append(f"temperatura muy alta ({row['t2m_max']:.0f}°C)")
    elif row['t2m_max'] > 28:
        factores.append(f"temperatura elevada ({row['t2m_max']:.0f}°C)")

    # Días sin lluvia
    if row['dias_sin_lluvia'] > 15:
        factores.append(f"sequía extrema ({row['dias_sin_lluvia']} días sin lluvia)")
    elif row['dias_sin_lluvia'] > 10:
        factores.append(f"sequía prolongada ({row['dias_sin_lluvia']} días sin lluvia)")
    elif row['dias_sin_lluvia'] > 5:
        factores.append(f"varios días sin lluvia ({row['dias_sin_lluvia']:.0f})")
    elif row['dias_sin_lluvia'] > 2:
        factores.append(f"sin lluvia en los últimos {row['dias_sin_lluvia']:.0f} días")

    # Viento
    if row['wind_max'] > 25:
        factores.append(f"viento muy fuerte ({row['wind_max']:.0f} km/h)")
    elif row['wind_max'] > 15:
        factores.append(f"viento fuerte ({row['wind_max']:.0f} km/h)")
    elif row['wind_max'] > 10:
        factores.append(f"viento moderado ({row['wind_max']:.0f} km/h)")
    elif row['wind_max'] > 8:
        factores.append(f"viento algo ventoso ({row['wind_max']:.0f} km/h)")

    # Suelo seco (balance hídrico)
    if row['swvl1_aprox'] < 0.12:
        factores.append("suelo extremadamente seco")
    elif row['swvl1_aprox'] < 0.15:
        factores.append("suelo seco")

    # Fuegos activos
    if row['fuegos_activos_lag1'] > 0:
        factores.append(f"ayer hubo {row['fuegos_activos_lag1']} foco activo en la zona")

    # Actividad humana
    if row['es_feriado'] == 1:
        factores.append("día feriado (mayor actividad humana)")
    elif row['es_finde_semana'] == 1:
        factores.append("fin de semana (mayor actividad humana)")

    # Vegetación
    if row['ndvi_anomalia_aprox'] < -0.15:
        factores.append("vegetación mucho más seca de lo normal")
    elif row['ndvi_anomalia_aprox'] < -0.08:
        factores.append("vegetación más seca de lo normal")

    # Rayos
    if row['tasa_rayos'] > 0.3:
        factores.append("alta probabilidad de rayos")
    elif row['tasa_rayos'] > 0.15:
        factores.append("probabilidad moderada de rayos")

    # Densidad poblacional
    if row['densidad_poblacional'] > 50:
        factores.append("zona muy poblada")
    elif row['densidad_poblacional'] > 20:
        factores.append("zona poblada")

    # Humedad relativa
    if 'hr_mean' in row and row['hr_mean'] < 30:
        factores.append(f"humedad muy baja ({row['hr_mean']:.0f}%)")
    elif 'hr_mean' in row and row['hr_mean'] < 40:
        factores.append(f"humedad baja ({row['hr_mean']:.0f}%)")

    # FWI (índice de peligro)
    if 'fwi' in row and row['fwi'] > 50:
        factores.append(f"índice de peligro extremo (FWI={row['fwi']:.0f})")
    elif 'fwi' in row and row['fwi'] > 30:
        factores.append(f"índice de peligro alto (FWI={row['fwi']:.0f})")

    if not factores:
        return "Condiciones climáticas dentro de parámetros normales, pero se recomienda vigilancia."
    else:
        texto = ", ".join(factores)
        return texto[0].upper() + texto[1:] + "."

def main():
    # ---- Cargar modelo y límite ----
    model_data   = cargar_modelo()
    model        = model_data['modelo']
    features     = model_data['features']
    limite_salta = cargar_limite_salta()

    # ---- Tablas auxiliares ----
    clim_t2m  = pd.read_parquet(ERA5_DIR / "climatologia_t2m.parquet")
    clim_ndvi = pd.read_parquet(ERA5_DIR / "climatologia_ndvi.parquet")
    dens_pob  = pd.read_parquet(ERA5_DIR / "densidad_poblacional_celdas.parquet")
    rayos_df  = pd.read_parquet(ERA5_DIR / "rayos_celdas.parquet")
    top_celdas = pd.read_parquet(ERA5_DIR / "top_celdas.parquet")

    # ---- Altitud de las celdas (estática) ----
    alt_celdas = pd.read_parquet(ERA5_DIR / "altitud_celdas.parquet")
    alt_dict = alt_celdas.set_index(['lat_era5', 'lon_era5'])['altitud'].to_dict()

    # ---- Cargar climatología de FWI (opcional) ----
    fwi_clim_path = ERA5_DIR / "fwi_climatologia.parquet"
    if fwi_clim_path.exists():
        fwi_clim = pd.read_parquet(fwi_clim_path)
    else:
        print("⚠️ No se encontró fwi_climatologia.parquet. Se usarán valores 0 para FWI/ISI/BUI.")
        fwi_clim = None

    # ---- Celdas monitoreadas: top celdas ----
    focos = top_celdas[['lat_era5', 'lon_era5']].rename(columns={'lat_era5': 'lat', 'lon_era5': 'lon'})
    focos = filtrar_en_salta(focos, limite_salta)
    focos = focos.rename(columns={'lat': 'lat_era5', 'lon': 'lon_era5'})
    depto_map = top_celdas.set_index(['lat_era5', 'lon_era5'])['departamento'].to_dict()
    print(f"Monitoreando {len(focos)} celdas prioritarias")

    # ---- Último NDVI ----
    fired = pd.read_parquet(ERA5_DIR / "fired_completo.parquet")
    fired_hist = fired[['lat_era5', 'lon_era5', 'date', 'ndvi']].rename(columns={'date': 'fecha'})
    neg_hist   = pd.read_parquet(ERA5_DIR / "negativos.parquet")[['lat_era5', 'lon_era5', 'fecha', 'ndvi']]
    df_hist    = pd.concat([fired_hist, neg_hist], ignore_index=True)
    df_hist['fecha'] = pd.to_datetime(df_hist['fecha'])
    fecha_hoy  = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    ultimo_ndvi = (
        df_hist[df_hist['fecha'] < fecha_hoy]
        .groupby(['lat_era5', 'lon_era5'])
        .last()['ndvi']
        .reset_index()
        .rename(columns={'ndvi': 'ndvi_obs'})
    )
    focos = focos.merge(ultimo_ndvi, on=['lat_era5', 'lon_era5'], how='left')
    focos['ndvi_obs'] = focos['ndvi_obs'].fillna(0.3)

    # ---- Densidad poblacional ----
    focos = focos.merge(dens_pob, on=['lat_era5', 'lon_era5'], how='left')
    focos['densidad_poblacional'] = focos['densidad_poblacional'].fillna(0)

    # ---- FIRMS (focos activos hoy) ----
    bbox_salta = [-68.5, -26.5, -62.0, -21.5]
    focos_activos = obtener_fuegos_activos(bbox_salta)
    celdas_era5 = focos[['lat_era5', 'lon_era5']].drop_duplicates()
    fuegos_x_celda = asignar_fuegos_a_celdas(focos_activos, celdas_era5)
    focos = focos.merge(fuegos_x_celda, on=['lat_era5', 'lon_era5'], how='left')
    focos['fuegos_activos_lag1'] = focos['fuegos_activos_lag1'].fillna(0).astype(int)
    print(f"Celdas con focos activos hoy: {(focos['fuegos_activos_lag1'] > 0).sum()}")

    # ---- Fecha pronóstico ----
    fecha_alerta = datetime.now().date() + timedelta(days=1)
    mes_alerta = fecha_alerta.month
    dia_año = fecha_alerta.timetuple().tm_yday
    print(f"Consultando pronóstico para {fecha_alerta}...")

    # ---- Feriados ----
    anio_alerta = fecha_alerta.year
    url_feriados = f"https://date.nager.at/api/v3/PublicHolidays/{anio_alerta}/AR"
    try:
        resp = requests.get(url_feriados, timeout=10)
        resp.raise_for_status()
        holidays_set = set(h['date'] for h in resp.json())
    except:
        print("⚠️ No se pudieron obtener feriados.")
        holidays_set = set()

    # ---- Obtener presión en la cordillera (UNA SOLA VEZ, fuera del bucle) ----
    print("Obteniendo presión en la cordillera...")
    try:
        presion_cordillera_hpa = obtener_presion_cordillera(fecha_alerta)
    except Exception as e:
        print(f"⚠️ Error al obtener presión en cordillera: {e}. Se usará 0.")
        presion_cordillera_hpa = 0

    # ---- Obtener presión en Chile (UNA SOLA VEZ, fuera del bucle) ----
    print("Obteniendo presión en Chile...")
    try:
        presion_chile_hpa = obtener_presion_chile(fecha_alerta)
    except Exception as e:
        print(f"⚠️ Error al obtener presión en Chile: {e}. Se usará 0.")
        presion_chile_hpa = 0

    # ---- Procesar cada celda ----
    todos = []
    total = len(focos)
    for i, (_, row) in enumerate(focos.iterrows()):
        if i % 20 == 0:
            print(f"  Procesando... {i}/{total}")

        try:
            # Obtener pronóstico para el día objetivo (mañana)
            data_fc = obtener_pronostico_dia_objetivo(row['lat_era5'], row['lon_era5'], fecha_alerta)
            celda = data_fc['daily']
        except Exception as e:
            print(f"  ⚠️ Error pronóstico ({row['lat_era5']}, {row['lon_era5']}): {e}")
            continue

        # --- Variables del pronóstico (día objetivo) ---
        t2m_max  = celda['temperature_2m_max'][0]
        t2m_mean = celda['temperature_2m_mean'][0]
        hr_mean  = celda['relative_humidity_2m_mean'][0]
        wind_max = celda['wind_speed_10m_max'][0] / 3.6
        wind_dir = celda['wind_direction_10m_dominant'][0]
        precip   = celda['precipitation_sum'][0]
        sp_min   = celda['surface_pressure_min'][0] * 100

      
        # --- Obtener datos de los 2 días anteriores para ventanas móviles (usando API de archivo + pronóstico) ---
        fecha_hoy = datetime.now().date()
        fecha_objetivo = fecha_alerta
        fechas_necesarias = [fecha_objetivo - timedelta(days=2), fecha_objetivo - timedelta(days=1), fecha_objetivo]
        
        t2m_max_list = []
        precip_list = []
        
        for fecha in fechas_necesarias:
            if fecha <= fecha_hoy:
                # Fecha pasada o hoy -> usar API de archivo
                try:
                    url_archivo = "https://archive-api.open-meteo.com/v1/archive"
                    params_ventana = {
                        "latitude": row['lat_era5'],
                        "longitude": row['lon_era5'],
                        "start_date": fecha.strftime("%Y-%m-%d"),
                        "end_date": fecha.strftime("%Y-%m-%d"),
                        "daily": "temperature_2m_max,precipitation_sum",
                        "timezone": "America/Argentina/Salta"
                    }
                    resp_ventana = requests.get(url_archivo, params=params_ventana, timeout=30)
                    resp_ventana.raise_for_status()
                    data_ventana = resp_ventana.json()['daily']
                    t2m_max_list.append(data_ventana['temperature_2m_max'][0])
                    precip_list.append(data_ventana['precipitation_sum'][0])
                except Exception as e:
                    print(f"  ⚠️ Error al obtener ventana móvil para fecha {fecha}: {e}. Usando día actual.")
                    t2m_max_list.append(t2m_max)
                    precip_list.append(precip)
            else:
                # Fecha futura (mañana) -> usar pronóstico
                t2m_max_list.append(t2m_max)
                precip_list.append(precip)
        
        # Calcular media y suma
        t2m_max_3d_avg = np.mean(t2m_max_list)
        precip_3d_sum = np.sum(precip_list)    

        # --- Presión en Chile y gradientes ---
        pressure_gradient = (presion_chile_hpa * 100) - sp_min
        pressure_gradient_zonda = (presion_chile_hpa - presion_cordillera_hpa) * 100
        altitud = alt_dict.get((row['lat_era5'], row['lon_era5']), 1000)

        # Obtener ráfaga máxima y humedad mínima de la celda
        wind_gust = celda['wind_gusts_10m_max'][0] / 3.6
        hr_min = celda['relative_humidity_2m_min'][0]

        # --- Histórico de precipitación (últimos 7 días reales) ---
        try:
            precip_hist = obtener_precipitacion_historica(row['lat_era5'], row['lon_era5'],
                                                          fecha_alerta - timedelta(days=7),
                                                          fecha_alerta - timedelta(days=1))
        except:
            precip_hist = [0.0] * 7
        if len(precip_hist) < 7:
            precip_hist = [0.0] * 7

        dias_sin_lluvia = 0
        for p in reversed(precip_hist):
            if p <= 0.1:
                dias_sin_lluvia += 1
            else:
                break
        precip_7d = sum(precip_hist)

        # Balance hídrico (usando datos de los últimos 7 días, pero solo tenemos precipitación, no temperatura)
        # Para simplificar, usamos el mismo método que en entrenamiento (requiere t2m_mean_7d y t2m_max_7d)
        # Como no tenemos esos datos diarios, usamos el día actual como aproximación (como antes)
        t2m_mean_7d = t2m_mean
        t2m_max_7d = t2m_max
        et_7d = 0.0023 * (t2m_mean_7d + 17.8) * max(t2m_max_7d - t2m_mean_7d, 0) * 7
        balance_hidrico_7d = precip_7d - et_7d
        swvl1_aprox = np.clip(0.1 + 0.005 * balance_hidrico_7d, 0.05, 0.35)

        # --- Anomalías y climatología ---
        norm_t = clim_t2m[(clim_t2m['lat_era5'] == row['lat_era5']) &
                          (clim_t2m['lon_era5'] == row['lon_era5']) &
                          (clim_t2m['mes'] == mes_alerta)]
        t2m_clim = norm_t['t2m_clim'].values[0] if len(norm_t) > 0 else t2m_mean
        t2m_anomalia = t2m_mean - t2m_clim

        norm_n = clim_ndvi[(clim_ndvi['lat_era5'] == row['lat_era5']) &
                           (clim_ndvi['lon_era5'] == row['lon_era5']) &
                           (clim_ndvi['mes'] == mes_alerta)]
        ndvi_clim = norm_n['ndvi_clim'].values[0] if len(norm_n) > 0 else 0.3
        ndvi_anomalia = row['ndvi_obs'] - ndvi_clim

        # --- Rayos ---
        mask_rayos = ((rayos_df['lat_era5'] == row['lat_era5']) &
                      (rayos_df['lon_era5'] == row['lon_era5']) &
                      (rayos_df['dia_año'] == dia_año))
        tasa_rayos_vals = rayos_df.loc[mask_rayos, 'tasa_rayos'].values
        tasa_rayos = tasa_rayos_vals[0] if len(tasa_rayos_vals) > 0 else 0

        # --- Feriado / fin de semana ---
        es_feriado = 1 if fecha_alerta.strftime('%Y-%m-%d') in holidays_set else 0
        es_finde   = 1 if fecha_alerta.weekday() >= 5 else 0

        # --- Otras features ---
        rayos_ndvi = tasa_rayos * max(ndvi_anomalia, 0)
        temp_ndvi = t2m_anomalia * max(ndvi_anomalia, 0)
        fire_danger = max(t2m_max - 20, 0) * (wind_max / 10) * (1 - swvl1_aprox)

        # --- temp_hr y FWI ---
        temp_hr = t2m_max * (100 - hr_mean) / 100
        if fwi_clim is not None:
            fwi_row = fwi_clim[(fwi_clim['lat_era5'] == row['lat_era5']) &
                               (fwi_clim['lon_era5'] == row['lon_era5']) &
                               (fwi_clim['mes'] == mes_alerta)]
            if len(fwi_row) > 0:
                fwi_val = fwi_row.iloc[0]['fwi']
                isi_val = fwi_row.iloc[0]['isi']
                bui_val = fwi_row.iloc[0]['bui']
            else:
                fwi_val, isi_val, bui_val = 0, 0, 0
        else:
            fwi_val, isi_val, bui_val = 0, 0, 0

        todos.append({
            'lat': row['lat_era5'],
            'lon': row['lon_era5'],
            't2m_max': t2m_max,
            't2m_mean': t2m_mean,
            't2m_anomalia_aprox': t2m_anomalia,
            'wind_max': wind_max,
            'wind_dir': wind_dir,
            'precip': precip,
            'dias_sin_lluvia': dias_sin_lluvia,
            'sp_min': sp_min,
            'swvl1_aprox': swvl1_aprox,
            'ndvi_lag15': row['ndvi_obs'],
            'ndvi_anomalia_aprox': ndvi_anomalia,
            'sin_dia': np.sin(2 * np.pi * dia_año / 365.25),
            'cos_dia': np.cos(2 * np.pi * dia_año / 365.25),
            'densidad_poblacional': row['densidad_poblacional'],
            'es_feriado': es_feriado,
            'es_finde_semana': es_finde,
            'tasa_rayos': tasa_rayos,
            't2m_max_3d_avg': t2m_max_3d_avg,
            'precip_3d_sum': precip_3d_sum,
            'rayos_ndvi': rayos_ndvi,
            'temp_ndvi': temp_ndvi,
            'fire_danger': fire_danger,
            'fuegos_activos_lag1': row['fuegos_activos_lag1'],
            'hr_mean': hr_mean,
            'temp_hr': temp_hr,
            'fwi': fwi_val,
            'isi': isi_val,
            'bui': bui_val,
            'departamento': depto_map.get((row['lat_era5'], row['lon_era5']), "Desconocido"),
            'pressure_gradient': pressure_gradient,
            'wind_gust_max_chile': wind_gust,
            'hr_min_chile': hr_min,
            'pressure_gradient_zonda': pressure_gradient_zonda,
            'altitud': altitud
        })

    df = pd.DataFrame(todos)

    # Verificar features del modelo y rellenar faltantes
    missing = set(features) - set(df.columns)
    if missing:
        print(f"⚠️ Features faltantes: {missing}, se rellenan con 0")
        for col in missing:
            df[col] = 0

    X = df[features]
    probas = model.predict_proba(X)[:, 1]
    df['probabilidad'] = probas
    df['alerta'] = (probas >= UMBRAL).astype(int)

    alertas = df[df['alerta'] == 1].copy()
    print(f"\n🔔 Alertas para {fecha_alerta}: {len(alertas)} celdas")

    if len(alertas) > 0:
        alertas_salta = filtrar_en_salta(alertas, limite_salta)
    else:
        alertas_salta = alertas.copy()

    # Guardar historial
    historial_dir = ERA5_DIR / "historial"
    historial_dir.mkdir(exist_ok=True)
    fecha_str = fecha_alerta.strftime('%Y-%m-%d')
    if len(alertas_salta) > 0:
        alertas_salta['fecha_pronostico'] = fecha_str
        path_alertas = historial_dir / "historial_alertas.csv"
        if path_alertas.exists():
            old = pd.read_csv(path_alertas)
            new = pd.concat([old, alertas_salta[['fecha_pronostico', 'lat', 'lon', 'probabilidad', 'departamento']]], ignore_index=True)
        else:
            new = alertas_salta[['fecha_pronostico', 'lat', 'lon', 'probabilidad', 'departamento']]
        new.to_csv(path_alertas, index=False)
        print(f"💾 Historial de alertas actualizado: {len(alertas_salta)} nuevas alertas")

    # Mensaje Telegram
    if len(alertas_salta) > 0:
        prob_prom = alertas_salta['probabilidad'].mean()
        top20 = alertas_salta.nlargest(20, 'probabilidad').reset_index(drop=True)
        top20_text = ""
        for i, row in top20.iterrows():
            if i < 5:
                explicacion = explicar_riesgo(row)
                top20_text += f"   {i+1}. {row['departamento']} ({abs(row['lat']):.2f}°S, {abs(row['lon']):.2f}°O) — {row['probabilidad']:.1%}\n      {explicacion}\n"
            else:
                top20_text += f"   {i+1}. {row['departamento']} ({abs(row['lat']):.2f}°S, {abs(row['lon']):.2f}°O) — {row['probabilidad']:.1%}\n"
        mensaje = (
            f"<b>🔥 ALERTA DE INCENDIOS FORESTALES</b>\n\n"
            f"📅 <b>Pronóstico:</b> {fecha_alerta.strftime('%d/%m/%Y')}\n"
            f"📍 <b>Zonas en riesgo:</b> {len(alertas_salta)} de {len(focos)}\n"
            f"📊 <b>Prob. promedio:</b> {prob_prom:.1%}\n\n"
            f"<b>Top 20 zonas críticas</b> (ver mapa):\n{top20_text}\n"
            f"<i>Los números ① a ⑤ en el mapa corresponden al Top 5.</i>"
        )
        alertas_salta['explicacion'] = alertas_salta.apply(explicar_riesgo, axis=1)
        mapa_path = generar_mapa(alertas_salta, limite_salta, fecha_alerta)
    else:
        mensaje = (
            f"<b>✅ INFORME DE RIESGO DE INCENDIOS</b>\n\n"
            f"📅 <b>Pronóstico:</b> {fecha_alerta.strftime('%d/%m/%Y')}\n"
            f"✅ <b>Sin zonas con riesgo significativo.</b>\n\n"
            f"<i>El monitoreo continúa. Próximo informe mañana.</i>"
        )
        mapa_path = generar_mapa(pd.DataFrame(), limite_salta, fecha_alerta)

    enviar_telegram_con_mapa(mensaje, mapa_path)
    print("\n✅ Proceso completado.")
    
    
if __name__ == "__main__":
    main()

## 📊 Performance Evaluation & Validation

This section evaluates the operational performance of the alerting system by comparing issued alerts against historical wildfire occurrences.

The evaluation framework measures the system's ability to correctly identify wildfire events while minimizing false alarms, providing both global and regional performance indicators.

Evaluation metrics include:

- Precision
- Recall (Sensitivity)
- F1 Score
- Daily Accuracy
- Specificity
- Department-level performance analysis

Results are automatically summarized and distributed via Telegram, enabling continuous monitoring and iterative model improvement.

The goal is to quantify the effectiveness of the early warning system and support evidence-based refinement of prediction and alerting strategies.

## Evaluation

## 🧪 How to proceed?

- Run periodically (e.g., every week or every month) to obtain a report on the system's real‑world performance.

- The report will be automatically sent to the configured Telegram chat.

- Results are also saved as CSV files for later analysis.

- The generated CSV files (`aciertos_detalle.csv`, `falsas_alarmas_detalle.csv`, `evaluacion_rendimiento.csv`, `evaluacion_por_departamento.csv`) allow more in‑depth analysis.

In [ ]:
"""
Evaluación del rendimiento del sistema de alerta.
Compara alertas emitidas con incendios reales (fuegos_activos_historicos.parquet).
Calcula métricas globales y por departamento.
Incluye verdaderos negativos (días sin alerta y sin incendios).
Envía resumen por Telegram.
"""

import pandas as pd
import numpy as np
import requests
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

ERA5_DIR = Path("era5_salta")
TELEGRAM_TOKEN = # "Insert your own token here"
TELEGRAM_CHAT_ID = # "your chat ID"

def enviar_telegram(mensaje):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": mensaje, "parse_mode": "HTML"}
    try:
        r = requests.post(url, json=payload, timeout=10)
        r.raise_for_status()
        print("✅ Mensaje enviado")
    except Exception as e:
        print(f"❌ Error: {e}")

def evaluar():
    # Cargar alertas
    path_alertas = ERA5_DIR / "historial" / "historial_alertas.csv"
    if not path_alertas.exists():
        print("No hay historial de alertas.")
        return
    alertas = pd.read_csv(path_alertas)
    alertas['fecha_pronostico'] = pd.to_datetime(alertas['fecha_pronostico'])
    
    # Cargar incendios reales (solo filas con fuegos_activos > 0)
    reales = pd.read_parquet(ERA5_DIR / "fuegos_activos_historicos.parquet")
    reales = reales[reales['fuegos_activos'] > 0]
    reales['fecha'] = pd.to_datetime(reales['fecha'])
    
    # Obtener top celdas para tener departamento
    top_celdas = pd.read_parquet(ERA5_DIR / "top_celdas.parquet")
    depto_map = top_celdas.set_index(['lat_era5', 'lon_era5'])['departamento'].to_dict()
    
    # Agregar departamento a alertas si no existe
    if 'departamento' not in alertas.columns:
        alertas['departamento'] = alertas.apply(lambda row: depto_map.get((row['lat'], row['lon']), "Desconocido"), axis=1)
    
    # ---- NUEVO: Definir el período de evaluación completo ----
    # Desde la primera fecha con alerta hasta ayer (porque hoy aún no se sabe)
    primera_fecha = alertas['fecha_pronostico'].min().date()
    fecha_fin = datetime.now().date() - timedelta(days=1)
    if primera_fecha > fecha_fin:
        print("No hay fechas para evaluar (el período es vacío).")
        return
    
    fechas_totales = pd.date_range(primera_fecha, fecha_fin, freq='D')
    
    # Preparar conjuntos de alertas por fecha (para acceso rápido)
    alertas_por_fecha = {}
    for fecha, grupo in alertas.groupby(alertas['fecha_pronostico'].dt.date):
        alertas_por_fecha[fecha] = set(zip(grupo['lat'], grupo['lon']))
    
    # Preparar incendios por fecha
    incendios_por_fecha = {}
    for fecha, grupo in reales.groupby(reales['fecha'].dt.date):
        incendios_por_fecha[fecha] = set(zip(grupo['lat_era5'], grupo['lon_era5']))
    
    # Listas para métricas diarias y por departamento
    resultados = []                     # para guardar día a día
    aciertos_por_depto = defaultdict(int)
    alertas_por_depto = defaultdict(int)
    incendios_por_depto = defaultdict(int)
    
    # Listas para detalles
    aciertos_list = []      # filas completas de aciertos
    falsas_list = []        # filas completas de falsas alarmas
    aciertos_detalle = []   # tuplas (fecha, depto, lat, lon)
    falsos_detalle = []     # tuplas
    
    # ---- Contadores globales a nivel de día ----
    VP_dias = 0   # días con alerta y con incendio
    FP_dias = 0   # días con alerta y sin incendio
    FN_dias = 0   # días sin alerta y con incendio
    VN_dias = 0   # días sin alerta y sin incendio
    
    # ---- Iterar sobre todas las fechas del período ----
    for fecha in fechas_totales:
        fecha_date = fecha.date()
        alertas_set = alertas_por_fecha.get(fecha_date, set())
        reales_set = incendios_por_fecha.get(fecha_date, set())
        
        # Clasificación a nivel de día
        tiene_alerta = len(alertas_set) > 0
        tiene_incendio = len(reales_set) > 0
        
        if tiene_alerta and tiene_incendio:
            VP_dias += 1
        elif tiene_alerta and not tiene_incendio:
            FP_dias += 1
        elif not tiene_alerta and tiene_incendio:
            FN_dias += 1
        else:
            VN_dias += 1
        
        # Si no hay alertas este día, no hay más que procesar a nivel de celdas
        if not tiene_alerta:
            continue
        
        # Para días con alertas, evaluamos celda por celda (igual que antes)
        aciertos_set = alertas_set & reales_set
        falsas_set = alertas_set - reales_set
        
        # Guardar filas completas para análisis
        alertas_dia = alertas[alertas['fecha_pronostico'].dt.date == fecha_date]
        for lat, lon in aciertos_set:
            row = alertas_dia[(alertas_dia['lat'] == lat) & (alertas_dia['lon'] == lon)].iloc[0]
            aciertos_list.append(row.to_dict())
        for lat, lon in falsas_set:
            row = alertas_dia[(alertas_dia['lat'] == lat) & (alertas_dia['lon'] == lon)].iloc[0]
            falsas_list.append(row.to_dict())
        
        # Métricas del día (por celda)
        aciertos = len(aciertos_set)
        falsas = len(falsas_set)
        total_alertas = len(alertas_set)
        total_incendios = len(reales_set)
        precision = aciertos / total_alertas if total_alertas > 0 else 0
        recall = aciertos / total_incendios if total_incendios > 0 else 0
        
        resultados.append({'fecha': fecha_date, 'alertas': total_alertas, 'aciertos': aciertos,
                           'falsas': falsas, 'precision': precision, 'recall': recall})
        
        # Por departamento
        for lat, lon in aciertos_set:
            depto = depto_map.get((lat, lon), "Desconocido")
            aciertos_por_depto[depto] += 1
            aciertos_detalle.append((fecha_date, depto, lat, lon))
        for lat, lon in falsas_set:
            depto = depto_map.get((lat, lon), "Desconocido")
            alertas_por_depto[depto] += 1
            falsos_detalle.append((fecha_date, depto, lat, lon))
        for lat, lon in reales_set:
            depto = depto_map.get((lat, lon), "Desconocido")
            incendios_por_depto[depto] += 1
    
    # Guardar CSVs de detalles
    if aciertos_list:
        df_aciertos = pd.DataFrame(aciertos_list)
        df_aciertos.to_csv(ERA5_DIR / "aciertos_detalle.csv", index=False)
        print(f"✅ Aciertos guardados: {len(df_aciertos)} registros")
    if falsas_list:
        df_falsas = pd.DataFrame(falsas_list)
        df_falsas.to_csv(ERA5_DIR / "falsas_alarmas_detalle.csv", index=False)
        print(f"✅ Falsas alarmas guardadas: {len(df_falsas)} registros")
    
    # Ajustar alertas_por_depto (sumar aciertos)
    for depto in aciertos_por_depto:
        alertas_por_depto[depto] += aciertos_por_depto[depto]
    
    # Métricas globales (por celda, igual que antes)
    df_res = pd.DataFrame(resultados)
    total_aciertos = df_res['aciertos'].sum() if not df_res.empty else 0
    total_alertas = df_res['alertas'].sum() if not df_res.empty else 0
    # Incendios reales solo en los días con alertas (para mantener coherencia con métricas anteriores)
    incendios_totales = sum(len(incendios_por_fecha.get(f, set())) for f in fechas_totales if f in incendios_por_fecha)
    recall_global = total_aciertos / incendios_totales if incendios_totales > 0 else 0
    precision_global = total_aciertos / total_alertas if total_alertas > 0 else 0
    f1_global = 2 * (precision_global * recall_global) / (precision_global + recall_global) if (precision_global + recall_global) > 0 else 0
    
    # Métricas por departamento (igual)
    depto_stats = []
    for depto in set(list(aciertos_por_depto.keys()) + list(incendios_por_depto.keys()) + list(alertas_por_depto.keys())):
        aciertos_d = aciertos_por_depto.get(depto, 0)
        alertas_d = alertas_por_depto.get(depto, 0)
        incendios_d = incendios_por_depto.get(depto, 0)
        prec_d = aciertos_d / alertas_d if alertas_d > 0 else 0
        rec_d = aciertos_d / incendios_d if incendios_d > 0 else 0
        f1_d = 2 * (prec_d * rec_d) / (prec_d + rec_d) if (prec_d + rec_d) > 0 else 0
        depto_stats.append({'departamento': depto, 'alertas': alertas_d, 'incendios': incendios_d,
                            'aciertos': aciertos_d, 'precision': prec_d, 'recall': rec_d, 'f1': f1_d})
    df_depto = pd.DataFrame(depto_stats).sort_values('f1', ascending=False)
    
    # Métricas a nivel de día (nuevas)
    total_dias = VP_dias + FP_dias + FN_dias + VN_dias
    exactitud = (VP_dias + VN_dias) / total_dias if total_dias > 0 else 0
    especificidad = VN_dias / (VN_dias + FP_dias) if (VN_dias + FP_dias) > 0 else 0
    recall_dias = VP_dias / (VP_dias + FN_dias) if (VP_dias + FN_dias) > 0 else 0
    precision_dias = VP_dias / (VP_dias + FP_dias) if (VP_dias + FP_dias) > 0 else 0
    f1_dias = 2 * (precision_dias * recall_dias) / (precision_dias + recall_dias) if (precision_dias + recall_dias) > 0 else 0
    
    # --- Mensaje Telegram ---
    mensaje = (
        f"<b>📊 INFORME DE RENDIMIENTO</b>\n\n"
        f"<b>Período:</b> {primera_fecha} a {fecha_fin}\n"
        f"<b>Alertas emitidas (total celdas):</b> {total_alertas}\n"
        f"<b>Incendios reales (celdas con fuego):</b> {incendios_totales}\n"
        f"<b>Aciertos (celdas):</b> {total_aciertos}\n"
        f"<b>Precisión (por celda):</b> {precision_global:.1%}\n"
        f"<b>Recall (por celda):</b> {recall_global:.1%}\n"
        f"<b>F1 (por celda):</b> {f1_global:.3f}\n\n"
        f"<b>📅 Evaluación por día:</b>\n"
        f"   Días con alerta y con incendio (VP): {VP_dias}\n"
        f"   Días con alerta y sin incendio (FP): {FP_dias}\n"
        f"   Días sin alerta y con incendio (FN): {FN_dias}\n"
        f"   Días sin alerta y sin incendio (VN): {VN_dias}\n"
        f"   Exactitud diaria: {exactitud:.1%}\n"
        f"   Especificidad: {especificidad:.1%}\n\n"
        f"<b>Top 5 departamentos por F1 (por celda):</b>\n"
    )
    for i, row in df_depto.head(5).iterrows():
        mensaje += f"   {i+1}. {row['departamento']}: Precisión {row['precision']:.1%}, Recall {row['recall']:.1%}, F1 {row['f1']:.3f}\n"
    
    if aciertos_detalle:
        mensaje += f"\n<b>Ejemplos de aciertos (Top 3):</b>\n"
        for fecha, depto, lat, lon in aciertos_detalle[:3]:
            mensaje += f"   {fecha}: {depto} ({abs(lat):.2f}°S, {abs(lon):.2f}°O)\n"
    if falsos_detalle:
        mensaje += f"\n<b>Ejemplos de falsas alarmas (Top 3):</b>\n"
        for fecha, depto, lat, lon in falsos_detalle[:3]:
            mensaje += f"   {fecha}: {depto} ({abs(lat):.2f}°S, {abs(lon):.2f}°O)\n"
    
    mensaje += f"\n<i>El sistema detectó {recall_global:.0%} de los incendios (por celda) y acertó en {precision_global:.0%} de las alertas. En {VN_dias} días no hubo alertas ni incendios (aciertos de seguridad).</i>"
    
    enviar_telegram(mensaje)
    
    # Guardar resúmenes
    if not df_res.empty:
        df_res.to_csv(ERA5_DIR / "evaluacion_rendimiento.csv", index=False)
    df_depto.to_csv(ERA5_DIR / "evaluacion_por_departamento.csv", index=False)
    print("✅ Informes guardados en CSV.")

if __name__ == "__main__":
    evaluar()